In [1]:
from pathlib import Path
import solweig

PROJECT_ROOT = Path.cwd().parent
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"

SOLWEIG_INPUTS = DATA_INTERIM / "solweig"
SOLWEIG_CACHE = DATA_INTERIM / "solweig_cache"
SOLWEIG_OUTPUTS = PROJECT_ROOT / "outputs" / "solweig"

SOLWEIG_CACHE.mkdir(parents=True, exist_ok=True)
SOLWEIG_OUTPUTS.mkdir(parents=True, exist_ok=True)

In [2]:
surface = solweig.SurfaceData.prepare(
    dsm=SOLWEIG_INPUTS / "dsm.tif",
    cdsm=SOLWEIG_INPUTS / "cdsm.tif",
    dem=SOLWEIG_INPUTS / "dem.tif",
    working_dir=SOLWEIG_CACHE,
)

solweig.models.surface: Preparing surface data from GeoTIFF files...
solweig.models.surface_loading:   DSM: 426×426 pixels
solweig.models.surface_loading:   Extracted pixel size from DSM: 5.00 m
solweig.models.surface_loading:   CRS validated: ETRS89 / UTM zone 30N (EPSG:25830)
solweig.models.surface_loading:   ✓ Canopy DSM (CDSM) provided
solweig.models.surface_loading:   ✓ Ground elevation (DEM) provided
solweig.models.surface_loading:   → No TDSM provided - will auto-generate from CDSM (ratio=0.25)
solweig.models.surface_loading: Checking for preprocessing data...
solweig.models.surface_loading:   → No walls found in working_dir - will compute from DSM and cache
solweig.models.surface_loading:   → No SVF found in working_dir - will compute and cache
solweig.models.surface_loading: Computing spatial extent and resolution...
solweig.models.surface_loading:   Auto-computed extent from raster intersection: [237173.53465099988, 4141421.4097209056, 239303.53465099988, 4143551.4097209056]


Computing wall aspects: 100%|██████████| 180/180 [00:00<00:00, 2280890.39it/s]

solweig.io: No-data value is -9999.0, replacing with NaN
solweig.io: No-data value is -9999.0, replacing with NaN
solweig.models.surface_compute:   ✓ Walls computed and cached to /Users/modjrin1/Library/CloudStorage/OneDrive-AaltoUniversity/paper 4 - mobility spain/heat-conditioned-accessibility/data/interim/solweig_cache/walls/px5.000
solweig.models.surface_compute: Computing SVF from DSM/CDSM/TDSM...



[GPU] Shadow GPU context initialized successfully
Computing Sky View Factor: 100%|██████████| 153/153 [00:00<00:00, 676.07it/s]

solweig.models.precomputed: Saved SVF memmap cache to /Users/modjrin1/Library/CloudStorage/OneDrive-AaltoUniversity/paper 4 - mobility spain/heat-conditioned-accessibility/data/interim/solweig_cache/svf/px5.000/memmap (15 arrays)


solweig.models.surface_serialization:   ✓ SVF saved as /Users/modjrin1/Library/CloudStorage/OneDrive-AaltoUniversity/paper 4 - mobility spain/heat-conditioned-accessibility/data/interim/solweig_cache/svf/px5.000/svfs.zip (compressed)
solweig.models.surface_serialization:   ✓ Shadow matrices saved as /Users/modjrin1/Library/CloudStorage/OneDrive-AaltoUniversity/paper 4 - mobility spain/heat-conditioned-accessibility/data/interim/solweig_cache/svf/px5.000/shadowmats.npz (compressed)
solweig.models.surface_compute:   ✓ SVF computed and cached to /Users/modjrin1/Library/CloudStorage/OneDrive-AaltoUniversity/paper 4 - mobility spain/heat-conditioned-accessibility/data/interim/solweig_cache/svf/px5.000
solweig.models.surface:   Valid mask: all pixels valid
solweig.models.surface:   Crop: no trimming needed (valid bbox = full extent)
solweig.models.surface:   Cleaned rasters saved to /Users/modjrin1/Library/CloudStorage/OneDrive-AaltoUniversity/paper 4 - mobility spain/heat-conditioned-access

In [3]:
location = solweig.Location.from_surface(
    surface,
    utc_offset=2
)

print(location)

Location(latitude=37.3922074009849, longitude=-5.956841746027315, altitude=0.0, utc_offset=2)


In [4]:
from datetime import datetime

weather = solweig.Weather(
    datetime=datetime(2025, 7, 15, 15, 0),
    ta=40.0,
    rh=25.0,
    global_rad=850.0,
    ws=2.0,
)

weather

Weather(datetime=datetime.datetime(2025, 7, 15, 15, 0), ta=40.0, rh=25.0, global_rad=850.0, ws=2.0, pressure=1013.25, timestep_minutes=60.0, measured_direct_rad=None, measured_diffuse_rad=None, precomputed_sun_altitude=None, precomputed_sun_azimuth=None, precomputed_altmax=None, sun_altitude=0.0, sun_azimuth=0.0, sun_zenith=90.0, direct_rad=0.0, diffuse_rad=0.0, clearness_index=1.0, altmax=45.0)

In [5]:
warnings = solweig.validate_inputs(
    surface,
    location,
    [weather]
)

for warning in warnings:
    print(warning)

In [6]:
summary = solweig.calculate(
    surface=surface,
    weather=[weather],
    location=location,
    output_dir=SOLWEIG_OUTPUTS,
    outputs=["tmrt", "shadow"],
)

solweig.tiling: Resource-aware tile sizing (context=solweig): GPU budget=9,534,832,640 bytes, RAM=2,583,642,112 available of 17,179,869,184 total, max_tile_side=1797 px
solweig.timeseries: ============================================================
solweig.timeseries: Starting SOLWEIG timeseries calculation
solweig.timeseries:   Grid size: 426x426 pixels
solweig.timeseries:   Timesteps: 1
solweig.timeseries:   Period: 2025-07-15 15:00 -> 2025-07-15 15:00
solweig.timeseries:   Location: 37.39N, -5.96E
solweig.timeseries: ============================================================
solweig.timeseries: Pre-computing sun positions and radiation splits...
solweig.timeseries:   Pre-computed 1 timesteps in 0.0s


SOLWEIG timeseries:   0%|          | 0/1 [00:00<?, ?it/s]

solweig.timeseries:   SOLWEIG timeseries: pre-computing ground view factor (reused across 1 timestep)


[GPU] GVF GPU context initialized
[GPU] Anisotropic sky GPU context initialized
SOLWEIG timeseries: 100%|██████████| 1/1 [00:00<00:00,  2.74it/s]

solweig.timeseries: ============================================================
solweig.timeseries: Calculation complete: 1 timesteps processed
solweig.timeseries:   Total time: 0.4s (2.72 steps/s)
solweig.timeseries: ============================================================


In [7]:
print(summary)

TimeseriesSummary(tmrt_mean=array([[64.315796, 64.61499 , 64.828186, ..., 64.8743  , 64.66147 ,
        64.23456 ],
       [64.63022 , 64.998474, 65.29587 , ..., 65.383484, 65.12918 ,
        64.66147 ],
       [64.84335 , 65.29587 , 65.59247 , ..., 65.679474, 65.383484,
        64.8743  ],
       ...,
       [64.98941 , 65.45825 , 65.70276 , ..., 65.83035 , 65.5907  ,
        65.072296],
       [64.77661 , 65.192474, 65.40668 , ..., 65.5387  , 65.23761 ,
        64.802795],
       [64.3385  , 64.62097 , 64.93808 , ..., 65.072296, 64.86035 ,
        64.613495]], shape=(426, 426), dtype=float32), tmrt_max=array([[64.315796, 64.61499 , 64.828186, ..., 64.8743  , 64.66147 ,
        64.23456 ],
       [64.63022 , 64.998474, 65.29587 , ..., 65.383484, 65.12918 ,
        64.66147 ],
       [64.84335 , 65.29587 , 65.59247 , ..., 65.679474, 65.383484,
        64.8743  ],
       ...,
       [64.98941 , 65.45825 , 65.70276 , ..., 65.83035 , 65.5907  ,
        65.072296],
       [64.77661 , 65.19

In [8]:
print("Mean Tmrt:", summary.tmrt_mean.mean())
print("Max Tmrt:", summary.tmrt_max.max())

Mean Tmrt: 67.1667
Max Tmrt: 71.50845


In [9]:
for p in sorted(SOLWEIG_OUTPUTS.rglob("*")):
    if p.is_file():
        print(p.relative_to(SOLWEIG_OUTPUTS))

run_metadata.json
shadow/shadow_20250715_1500.tif
summary/shade_hours.preview.png
summary/shade_hours.tif
summary/sun_hours.preview.png
summary/sun_hours.tif
summary/tmrt_day_mean.preview.png
summary/tmrt_day_mean.tif
summary/tmrt_max.preview.png
summary/tmrt_max.tif
summary/tmrt_mean.preview.png
summary/tmrt_mean.tif
summary/tmrt_min.preview.png
summary/tmrt_min.tif
summary/tmrt_night_mean.tif
summary/utci_day_mean.preview.png
summary/utci_day_mean.tif
summary/utci_hours_above_26_night.preview.png
summary/utci_hours_above_26_night.tif
summary/utci_hours_above_32_day.preview.png
summary/utci_hours_above_32_day.tif
summary/utci_hours_above_38_day.preview.png
summary/utci_hours_above_38_day.tif
summary/utci_max.preview.png
summary/utci_max.tif
summary/utci_mean.preview.png
summary/utci_mean.tif
summary/utci_min.preview.png
summary/utci_min.tif
summary/utci_night_mean.tif
tmrt/tmrt_20250715_1500.tif


In [10]:
EPW_DIR = DATA_INTERIM / "weather"
EPW_DIR.mkdir(parents=True, exist_ok=True)

epw_path = solweig.download_epw(
    latitude=37.39,
    longitude=-5.96,
    output_path=EPW_DIR / "sevilla.epw",
)

print(epw_path)

solweig.io_epw: Downloading EPW from PVGIS for (37.3900, -5.9600)...
solweig.io_epw: Saved EPW file: /Users/modjrin1/Library/CloudStorage/OneDrive-AaltoUniversity/paper 4 - mobility spain/heat-conditioned-accessibility/data/interim/weather/sevilla.epw (8761 hourly records)
/Users/modjrin1/Library/CloudStorage/OneDrive-AaltoUniversity/paper 4 - mobility spain/heat-conditioned-accessibility/data/interim/weather/sevilla.epw


In [12]:
weather_list = solweig.Weather.from_epw(
    epw_path,
    start="07-15",
    end="07-15",
)

print("Timesteps:", len(weather_list))
print(weather_list[0])

solweig.io_epw: Loaded EPW file: unknown, 8760 timesteps (pure Python parser)
solweig.models.weather: Loaded 24 timesteps from EPW: 2009-07-15 00:00 → 2009-07-15 23:00
Timesteps: 24
Weather(datetime=datetime.datetime(2009, 7, 15, 0, 0), ta=22.87, rh=52.1, global_rad=0.0, ws=1.7, pressure=1007.0, timestep_minutes=60.0, measured_direct_rad=-0.0, measured_diffuse_rad=0.0, precomputed_sun_altitude=None, precomputed_sun_azimuth=None, precomputed_altmax=None, sun_altitude=0.0, sun_azimuth=0.0, sun_zenith=90.0, direct_rad=0.0, diffuse_rad=0.0, clearness_index=1.0, altmax=45.0)


In [13]:
location_epw = solweig.Location.from_epw(epw_path)

print(location_epw)

solweig.models.location: Location from EPW: unknown — 37.3900°N, -5.9600°E (UTC+1, 15m)
Location(latitude=37.39, longitude=-5.96, altitude=15.0, utc_offset=1.0)


In [14]:
for w in weather_list:
    print(
        w.datetime,
        "Ta:", w.ta,
        "RH:", w.rh,
        "Rad:", w.global_rad,
        "Wind:", w.ws
    )

2009-07-15 00:00:00 Ta: 22.87 RH: 52.1 Rad: 0.0 Wind: 1.7
2009-07-15 01:00:00 Ta: 21.87 RH: 55.65 Rad: 0.0 Wind: 1.2
2009-07-15 02:00:00 Ta: 21.24 RH: 55.55 Rad: 0.0 Wind: 1.0
2009-07-15 03:00:00 Ta: 21.0 RH: 55.45 Rad: 0.0 Wind: 1.0
2009-07-15 04:00:00 Ta: 20.39 RH: 53.35 Rad: 0.0 Wind: 1.2
2009-07-15 05:00:00 Ta: 19.15 RH: 55.05 Rad: 0.0 Wind: 1.5
2009-07-15 06:00:00 Ta: 18.29 RH: 56.95 Rad: 0.0 Wind: 1.7
2009-07-15 07:00:00 Ta: 18.15 RH: 59.05 Rad: 113.0 Wind: 1.7
2009-07-15 08:00:00 Ta: 21.34 RH: 51.75 Rad: 305.0 Wind: 1.8
2009-07-15 09:00:00 Ta: 24.1 RH: 41.0 Rad: 514.0 Wind: 2.3
2009-07-15 10:00:00 Ta: 26.63 RH: 33.7 Rad: 703.0 Wind: 2.3
2009-07-15 11:00:00 Ta: 28.97 RH: 27.6 Rad: 850.0 Wind: 2.1
2009-07-15 12:00:00 Ta: 30.89 RH: 22.55 Rad: 956.0 Wind: 1.8
2009-07-15 13:00:00 Ta: 32.46 RH: 19.1 Rad: 1007.0 Wind: 1.4
2009-07-15 14:00:00 Ta: 33.66 RH: 16.05 Rad: 991.0 Wind: 0.9
2009-07-15 15:00:00 Ta: 34.57 RH: 14.55 Rad: 936.0 Wind: 0.7
2009-07-15 16:00:00 Ta: 35.19 RH: 13.7 Rad: 

In [15]:
summary = solweig.calculate(
    surface=surface,
    weather=weather_list,
    location=location_epw,
    output_dir=SOLWEIG_OUTPUTS,
    outputs=["tmrt", "shadow"],
)

solweig.timeseries: ============================================================
solweig.timeseries: Starting SOLWEIG timeseries calculation
solweig.timeseries:   Grid size: 426x426 pixels
solweig.timeseries:   Timesteps: 24
solweig.timeseries:   Period: 2009-07-15 00:00 -> 2009-07-15 23:00
solweig.timeseries:   Location: 37.39N, -5.96E
solweig.timeseries: ============================================================
solweig.timeseries: Pre-computing sun positions and radiation splits...
solweig.timeseries:   Pre-computed 24 timesteps in 0.0s


SOLWEIG timeseries:   0%|          | 0/24 [00:00<?, ?it/s]

solweig.timeseries:   SOLWEIG timeseries: pre-computing ground view factor (reused across 24 timesteps)


SOLWEIG timeseries: 100%|██████████| 24/24 [00:02<00:00, 11.64it/s]

solweig.timeseries: ============================================================
solweig.timeseries: Calculation complete: 24 timesteps processed
solweig.timeseries:   Total time: 2.1s (11.61 steps/s)
solweig.timeseries: ============================================================


In [16]:
print("Timesteps:", summary.n_timesteps)
print("Mean Tmrt:", summary.tmrt_mean.mean())
print("Max Tmrt:", summary.tmrt_max.max())

Timesteps: 24
Mean Tmrt: 38.53337
Max Tmrt: 72.16629


In [17]:
ts = summary.timeseries

print(ts.datetime)
print(ts.ta)
print(ts.tmrt_mean)
print(ts.utci_mean)
print(ts.sun_fraction)

[datetime.datetime(2009, 7, 15, 0, 0), datetime.datetime(2009, 7, 15, 1, 0), datetime.datetime(2009, 7, 15, 2, 0), datetime.datetime(2009, 7, 15, 3, 0), datetime.datetime(2009, 7, 15, 4, 0), datetime.datetime(2009, 7, 15, 5, 0), datetime.datetime(2009, 7, 15, 6, 0), datetime.datetime(2009, 7, 15, 7, 0), datetime.datetime(2009, 7, 15, 8, 0), datetime.datetime(2009, 7, 15, 9, 0), datetime.datetime(2009, 7, 15, 10, 0), datetime.datetime(2009, 7, 15, 11, 0), datetime.datetime(2009, 7, 15, 12, 0), datetime.datetime(2009, 7, 15, 13, 0), datetime.datetime(2009, 7, 15, 14, 0), datetime.datetime(2009, 7, 15, 15, 0), datetime.datetime(2009, 7, 15, 16, 0), datetime.datetime(2009, 7, 15, 17, 0), datetime.datetime(2009, 7, 15, 18, 0), datetime.datetime(2009, 7, 15, 19, 0), datetime.datetime(2009, 7, 15, 20, 0), datetime.datetime(2009, 7, 15, 21, 0), datetime.datetime(2009, 7, 15, 22, 0), datetime.datetime(2009, 7, 15, 23, 0)]
[22.87 21.87 21.24 21.   20.39 19.15 18.29 18.15 21.34 24.1  26.63 28.97


In [18]:
import pandas as pd

hourly = pd.DataFrame({
    "datetime": ts.datetime,
    "ta": ts.ta,
    "tmrt_mean": ts.tmrt_mean,
    "utci_mean": ts.utci_mean,
    "sun_fraction": ts.sun_fraction,
})

hourly

,datetime,ta,tmrt_mean,utci_mean,sun_fraction
0,2009-07-15 00:00:00,22.870001,16.183928,19.676649,0.000000
1,2009-07-15 01:00:00,21.870001,15.227113,19.336277,0.000000
2,2009-07-15 02:00:00,21.240000,14.519658,18.818895,0.000000
3,2009-07-15 03:00:00,21.000000,14.247926,18.544680,0.000000
4,2009-07-15 04:00:00,20.389999,13.480135,17.514336,0.000000
5,2009-07-15 05:00:00,19.150000,12.176482,15.829352,0.000000
6,2009-07-15 06:00:00,18.290001,11.299716,14.648494,0.000000
7,2009-07-15 07:00:00,18.150000,19.237246,17.201382,0.263198
8,2009-07-15 08:00:00,21.340000,36.539307,24.322258,0.622461
9,2009-07-15 09:00:00,24.100000,48.844585,28.584881,0.781729


In [19]:
hourly.loc[hourly["utci_mean"].idxmax()]

datetime        2009-07-15 16:00:00
ta                        35.189999
tmrt_mean                 65.647102
utci_mean                 41.033802
sun_fraction               0.965952
Name: 16, dtype: object

In [20]:
hourly.loc[hourly["tmrt_mean"].idxmax()]

datetime        2009-07-15 16:00:00
ta                        35.189999
tmrt_mean                 65.647102
utci_mean                 41.033802
sun_fraction               0.965952
Name: 16, dtype: object

In [21]:
summary = solweig.calculate(
    surface=surface,
    weather=weather_list,
    location=location_epw,
    output_dir=SOLWEIG_OUTPUTS,
    outputs=["tmrt", "utci", "shadow"],
)

solweig.timeseries: ============================================================
solweig.timeseries: Starting SOLWEIG timeseries calculation
solweig.timeseries:   Grid size: 426x426 pixels
solweig.timeseries:   Timesteps: 24
solweig.timeseries:   Period: 2009-07-15 00:00 -> 2009-07-15 23:00
solweig.timeseries:   Location: 37.39N, -5.96E
solweig.timeseries: ============================================================
solweig.timeseries: Pre-computing sun positions and radiation splits...
solweig.timeseries:   Pre-computed 24 timesteps in 0.0s


SOLWEIG timeseries:   0%|          | 0/24 [00:00<?, ?it/s]

solweig.timeseries:   SOLWEIG timeseries: pre-computing ground view factor (reused across 24 timesteps)


SOLWEIG timeseries: 100%|██████████| 24/24 [00:02<00:00, 10.77it/s]

solweig.timeseries: ============================================================
solweig.timeseries: Calculation complete: 24 timesteps processed
solweig.timeseries:   Total time: 2.2s (10.75 steps/s)
solweig.timeseries: ============================================================


In [22]:
print(summary.report())

SOLWEIG Summary: 24 timesteps (15 day, 9 night)
  Period: 2009-07-15 00:00 — 2009-07-15 23:00
  Tmrt  — mean: 38.5°C, range: 10.6 – 72.2°C
  UTCI  — mean: 28.2°C, range: 14.4 – 42.8°C
  Sun   — 0.0 – 15.0 hours
  UTCI > 32°C (day) — max 11.0h
  UTCI > 38°C (day) — max 8.0h
  Ta    — range: 18.1 – 35.4°C
  Summary GeoTIFFs: /Users/modjrin1/Library/CloudStorage/OneDrive-AaltoUniversity/paper 4 - mobility spain/heat-conditioned-accessibility/outputs/solweig/summary/
    shade_hours.tif
    sun_hours.tif
    tmrt_day_mean.tif
    tmrt_max.tif
    tmrt_mean.tif
    tmrt_min.tif
    tmrt_night_mean.tif
    utci_day_mean.tif
    utci_hours_above_26_night.tif
    utci_hours_above_32_day.tif
    utci_hours_above_38_day.tif
    utci_max.tif
    utci_mean.tif
    utci_min.tif
    utci_night_mean.tif

Tip: per-timestep arrays are in summary.timeseries (e.g. .ta, .tmrt_mean, .utci_mean).
     Spatial grids are on the summary itself (e.g. .tmrt_mean, .utci_max).
     Summary grids are saved as GeoTI